In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
def collate_fn(batch):
    # unpacks the batch, and then zip creates separate tuples for images and targets
    images, targets = zip(*batch)

    images = torch.stack(images)
    targets = list(targets)

    return images, targets

In [4]:
from torch.utils.data import DataLoader

dl = DataLoader(transformed_dataset, batch_size=4, collate_fn=collate_fn)
dl

In [5]:
X_batch, y_batch = next(iter(dl))

X_batch.shape, len(y_batch)

(torch.Size([4, 3, 224, 224]), 4)

In [6]:
from src.utilities import pairwise_giou_cxcywh, cxcywh_to_xyxy

bbox_preds = torch.randn(100, 4)
class_preds = torch.randn(100, 21)

In [7]:
_, truth_labels = next(iter(transformed_dataset))
truth_boxes = truth_labels[:, -4:]
truth_boxes.shape, truth_boxes

(torch.Size([5, 4]),
 tensor([[0.5848, 0.7321, 0.1205, 0.3393],
         [0.4196, 0.8482, 0.1741, 0.2902],
         [0.0714, 0.8259, 0.1250, 0.3482],
         [0.5357, 0.6562, 0.1071, 0.2812],
         [0.5893, 0.5402, 0.0714, 0.0893]]))

In [8]:
giou = pairwise_giou_cxcywh(truth_boxes, truth_boxes)

In [9]:
giou

tensor([[ 1.0000, -0.3209, -0.6967,  0.1593, -0.0323],
        [-0.3209,  1.0000, -0.4574, -0.3316, -0.6091],
        [-0.6967, -0.4574,  1.0000, -0.7380, -0.8394],
        [ 0.1593, -0.3316, -0.7380,  1.0000, -0.1367],
        [-0.0323, -0.6091, -0.8394, -0.1367,  1.0000]])

In [10]:
from src.matching import compute_cls_cost, compute_l1_cost, compute_giou_cost

cls_cost = compute_cls_cost(class_preds, truth_labels[:, 0])
l1_cost = compute_l1_cost(bbox_preds, truth_boxes)
giou_cost = compute_giou_cost(bbox_preds, truth_boxes)

cls_cost.shape, l1_cost.shape, giou_cost.shape

(torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]))

In [11]:
cls_cost[0], l1_cost[0], giou_cost[0]

(tensor([-0.0441, -0.0441, -0.0441, -0.0441, -0.0441]),
 tensor([3.7511, 3.7065, 3.3449, 3.5547, 3.2645]),
 tensor([0.9242, 0.9053, 0.8493, 0.9273, 0.9730]))

In [12]:
(cls_cost + l1_cost + giou_cost)[0]

tensor([4.6313, 4.5678, 4.1501, 4.4380, 4.1934])

In [13]:
1.0*cls_cost[0], 5.0*l1_cost[0], 2.0*giou_cost[0]

(tensor([-0.0441, -0.0441, -0.0441, -0.0441, -0.0441]),
 tensor([18.7557, 18.5325, 16.7244, 17.7735, 16.3226]),
 tensor([1.8484, 1.8107, 1.6986, 1.8546, 1.9459]))

In [14]:
(1.0*cls_cost + 5.0*l1_cost + 2.0*giou_cost)[0]

tensor([20.5600, 20.2991, 18.3789, 19.5841, 18.2245])

In [15]:
from src.matching import hungarian_match_costs

costs = hungarian_match_costs(class_preds, bbox_preds, truth_labels)

costs.shape, costs[0]

(torch.Size([100, 5]), tensor([20.5600, 20.2991, 18.3789, 19.5841, 18.2245]))

In [19]:
from scipy.optimize import linear_sum_assignment

pred_idxs, gt_idxs = linear_sum_assignment(costs)
pred_idxs, gt_idxs

(array([ 1, 28, 44, 69, 80]), array([2, 1, 4, 3, 0]))

In [23]:
target_matches = torch.full((100,), 20.0)

target_matches

tensor([20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20.])

In [27]:
target_matches[pred_idxs] = truth_labels[gt_idxs, 0]

target_matches

tensor([20.,  8., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
         8., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20.,  8., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,  8.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,  8., 20., 20., 20.,
        20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.,
        20., 20.])